# Echolocating bats homing over kilometer scales: circular stats on goal-relative bearings

This notebook revisits the free-flying bat homing data of **Goldshtein et al. (2024, *Science*)**
(*"Acoustic cognitive map--based navigation in echolocating bats"*: echolocating bats translocated over
kilometer scales and tracked back to a known goal). The biological
question is whether sensory deprivation (temporary occlusion of vision, disruption of
magnetoreception / olfaction) produces detectable changes in **goal-relative orientation**
(directional bias and/or directional precision), and how robust homing is when some sensory
modalities are degraded.

Each flight is a GPS track recorded with an on-animal tag. We reduce each track to a sequence of
fixed-length movement **steps**, compute a heading per step, and express that heading **relative
to the instantaneous direction toward the animal's known destination** (its roost, or the
Kinneret/Agamon lake region). On this goal-relative scale, a step aimed straight at the goal is
near $0^\circ$ and a step heading away is near $\pm180^\circ$, so fitted von Mises parameters map
directly onto goal-directed travel.

**Why a mixture:** stepwise headings in these tracks are multimodal. A bat may wander before
committing to goal-directed travel, and the original study had to split trajectories into a
localization phase and a homing phase using *post-treatment* features (straightness,
distance-to-goal). Selecting on such features can entangle treatment effects with the selection
rule. To avoid that, the main analysis below fits a **von Mises--Uniform mixture (vMU)**: each step
is drawn either from a directed von Mises component (goal-directed "exploitation") or from an
undirected component (wandering "exploration"), with treatment and per-animal (tag) random
effects. This separates trajectory structure from treatment comparisons while using the **full**
trajectory, with no post-treatment filtering.

The notebook also fits a simpler **single von Mises treatment model** (Bambi) first, so the vMU
result can be read against a non-mixed baseline, and compares the two out-of-sample (PSIS-LOO
ELPD).

**Environment (run once from the repository root, in a terminal):**

```bash
uv venv --python 3.12
uv pip install -e ".[notebook]" "pandas[excel]" geopandas shapely contextily osmnx pyproj xlrd
uv run --no-project jupyter lab notebooks/goldshtein_bats.ipynb
```

Then select that kernel and run the cells top to bottom. This notebook loads its
track/metadata tables and the Agamon–Hula lake outline from the public [jgraving/unwrap-data](https://github.com/jgraving/unwrap-data) repository via `unwrap.datasets` (`load_bats_tracks`, `load_bats_info`, `bats_lake_outline_path`); the first run clones it to `~/.cache/unwrap-data`,
after which no further download happens.


In [ ]:
%matplotlib inline
from importlib.metadata import version
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

import unwrap
from unwrap.datasets import load_bats_info, load_bats_tracks, bats_lake_outline_path

# The preprocessing, post-hoc and figure code lives in sibling scripts next
# to this notebook (goldshtein_bats_preprocess.py,
# goldshtein_bats_posthoc.py, goldshtein_bats_plots.py); make them
# importable regardless of the kernel's working directory (the kernel may
# run from notebooks/ or from the repository root).
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(unwrap.__file__)), "notebooks"))

import goldshtein_bats_preprocess as pre
import goldshtein_bats_posthoc as posthoc
import goldshtein_bats_plots as plots

plots.set_style()

SEED = 123
{package: version(package) for package in (
    "unwrap", "pymc", "arviz", "bambi",
    "numpy", "pandas", "scipy", "matplotlib",
)}


## 1. Load the data

Per-tag metadata and the raw track file. `meta_df` carries the treatment assignment and the
destination flags; `raw_track_df` carries the georeferenced track per tag.


In [ ]:
meta_df = load_bats_info()
meta_df

In [ ]:
raw_track_df = load_bats_tracks()
raw_track_df

## 2. Spatially resample to fixed step length

`rediscretize_latlon_parallel` resamples each tag's lat/lon trajectory to a **constant step length**
(60 m) along-track, computing step headings and distances. Gaps in the raw track (a jump larger than
`gap_factor \times spacing_m`, or a time gap larger than `gap_time_s`) split the track into
contiguous `segment_id`s, and the first resampled row after a gap is flagged `gap_before=1`. Working
with fixed step lengths (rather than variable GPS fix intervals) is what lets us treat each step's
bearing as one circular observation and keeps distance along-track `s_along_m` well defined.


In [ ]:
# Resample every tag's lat/lon trajectory to a constant 60 m step length;
# gaps (a jump > gap_factor * spacing_m = 600 m, or a time gap > 2 min)
# split the track into contiguous segments flagged by ``gap_before``.
rediscretized_ll_df = pre.rediscretize_latlon_parallel(
    raw_track_df,
    spacing_m=60.0,
    n_workers=16,
    include_last=False,
    gap_factor=10.0,
    gap_time_s=120.0,
    time_unit="days",
)

print(rediscretized_ll_df.head())
print("rows:", len(rediscretized_ll_df), "tags:", rediscretized_ll_df["tag"].nunique())
print("n segments total:", rediscretized_ll_df["segment_id"].nunique() if "segment_id" in rediscretized_ll_df else None)
print("gap markers:", int(rediscretized_ll_df["gap_before"].sum()) if "gap_before" in rediscretized_ll_df else None)

step_stats = pre.check_step_lengths_latlon(rediscretized_ll_df)
print(step_stats.describe(include="all"))


### Per-tag metadata + destination label

Collapse the metadata to one row per tag, coerce the destination flags to 0/1, and build a
`destination` label: `Roost` (flyHome) vs `Agamon` (flyAgamon, the Kinneret/lake region) vs
Unknown. The per-tag metadata are merged back onto the resampled tracks so every step knows its
treatment and goal.


In [ ]:
# One metadata row per tag (treatment + destination label), merged back
# onto the resampled steps so every step knows its treatment and goal.
rediscretized_ll_w_meta_df = pre.merge_tag_metadata(rediscretized_ll_df, meta_df)

print("resampled rows:", len(rediscretized_ll_w_meta_df))
print("missing treatment rows:",
      rediscretized_ll_w_meta_df["treatment"].isna().sum(),
      " (tags missing:",
      rediscretized_ll_w_meta_df.loc[rediscretized_ll_w_meta_df["treatment"].isna(), "tag"].nunique(),
      ")"
)


In [ ]:
rediscretized_ll_w_meta_df

## 3. Goal-relative bearings (distance + bearing to the destination)

For every resampled step, compute:

- its local metric coordinates (`x_m, y_m`, a projected plane around the arena),
- its step `bearing_to_...`/`direction`,
- the **instantaneous direction toward the goal**: for the lake (Agamon) this is the
  negative-gradient of a smoothed-distance (or harmonic) distance field (`-\nabla u`), so the
  "best way to reduce distance" from each point, not just a straight line to a single point; for
  the roost it is the bearing to the roost point,
- the **goal-relative angle** `flightAngDiffDest_{deg,rad}`: the animal's step bearing **relative to**
  that goal direction. This is the circular response used in the models below (0 = aimed at goal).

This is what places every flight on a common circular scale: steps toward the goal lie near $0^\circ$,
steps away near $\pm180^\circ$.


In [ ]:
# Distance + bearing to each animal's destination (lake: negative
# gradient of a harmonic distance field; roost: bearing to the roost
# point), plus the goal-relative angle ``flightAngDiffDest`` used as the
# circular response (0 deg = aimed at the goal).
df_harm = pre.add_dist_and_angle_to_destination(
    rediscretized_ll_w_meta_df,
    LAKE_GEOJSON=bats_lake_outline_path(),
    METRIC_EPSG=32636,
    ROOST_LAT=33.112373,
    ROOST_LON=35.584135,
    lake_grid_res_m=20.0,
    lake_domain_margin_m=None,
    lake_jacobi_tol=1e-5,
    lake_field="harmonic",
)
df_harm[
    ["tag", "dest",
     "dist_to_lake_m", "dist_to_lake_grad_m",
     "dist_to_dest_m", "dist_to_dest_grad_m",
     "step_len_m", "step_bearing_deg",
     "bearing_to_dest_deg", "flightAngDiffDest_deg"]
].head(10)

rediscretized_ll_w_meta_df2 = df_harm


In [ ]:
rediscretized_ll_w_meta_df2

## 4. Modeling table and descriptive summaries

Assemble the modeling frame and check the design.


In [ ]:
track_w_treat = rediscretized_ll_w_meta_df2.copy()

Observations and animals per treatment, then steps per animal, so we can see the repeated-measures
structure (many steps within each tag) that motivates the per-tag random effects.


In [ ]:
combo_summary = (
    track_w_treat.groupby(["treatment", "treatmentName"], dropna=False)
                 .agg(
                     n_rows=("tag", "size"),       # total trajectory rows/steps
                     n_tags=("tag", "nunique"),    # number of animals/tags
                 )
                 .sort_values(["n_tags", "n_rows"], ascending=False)
                 .reset_index()
)

combo_summary


In [ ]:
steps_per_tag = (
    track_w_treat.groupby(["treatment", "treatmentName", "tag"], dropna=False)
                 .size()
                 .rename("n_steps")
                 .reset_index()
)

steps_summary = (
    steps_per_tag.groupby(["treatment", "treatmentName"], dropna=False)["n_steps"]
                 .agg(n_tags="count", mean="mean", median="median", min="min", max="max")
                 .sort_values("n_tags", ascending=False)
                 .reset_index()
)

steps_summary


## 5. Trajectory maps (goal-relative context)

Panels of the resampled tracks, colored by destination (roost vs Agamon) and by treatment, with the
lake polygon (Agamon) overlaid. Useful ground truth for what "aimed at the goal" looks like in each
condition before we fit any model.


Full-track panels (all steps) by treatment \times destination.


In [ ]:
# Full-track destination panels by treatment x destination (basemap at
# zoom 15 with retina tiles, no repeated treatment legend).
plots.plot_destination_panels(
    track_w_treat,
    meta_df,
    basemap_zoom=15,
    basemap_retina=True,
    show_treatment_legend=False,
    save_path="figures/bats/rediscretized_destination_panels.pdf",
)
plt.show()


Compact per-panel trajectory plots (one panel per treatment / destination combination).


In [ ]:
# Compact version: one panel per treatment / destination combination,
# small figure with larger relative text (trajectory layers rasterized
# inside the saved PDF).
plots.plot_destination_panels_compact(
    track_w_treat,
    meta_df,
    fig_w=2.5,
    fig_h=3.8,
    title_fs=9,
    legend_fs=6,
    save_path="figures/bats/rediscretized_destination_panels_compact.pdf",
)
plt.show()


Per-treatment destination panels (compact).


After-localisation tracking and destination panels.


Destination flow fields: smoothed-distance and harmonic distance fields over the arena, with the
negative-gradient (goal-approach) direction field that supplies `bearing_to_dest` in the goal-relative
constructions above.


In [ ]:
# Goal-approach direction fields feeding bearing_to_dest: the
# smoothed-distance field over the lake panel and the radial-to-roost
# field over the roost panel (scale/compass UI off for the export).
plots.plot_destination_flow_fields(
    track_w_treat,
    meta_df,
    lake_field="smoothed_distance",
    lake_grid_res_m=200.0,
    lake_smooth_sigma_pix=1.0,
    roost_grid_res_m=200.0,
    quiver_scale_lake=30,
    quiver_scale_roost=30,
    show_ui_scale_compass=False,
    save_path="figures/bats/destination_flow_fields_smoothed_distance.pdf",
)
plt.show()


### Descriptive circular summaries by treatment

Per-treatment concentration summaries (descriptive, no model): estimated population concentration
($\hat\kappa$), mean resultant length ($\vec{R}$), and the alignment of each individual's mean
bearing to $0^\circ$ (goal direction). These give a first, model-free read on whether treatment
changes directional bias or precision.


In [ ]:
# Polar KDE of the goal-relative bearings by treatment x destination
# (von Mises kernel, kappa = 15).
plots.plot_per_tag_polar_kde(
    track_w_treat,
    kappa=15,
    r0=1.0,
    r_scale=1.0,
    save_path="figures/bats/polar_vonmises_kde_all.pdf",
)
plt.show()


In [ ]:
# Per-tag descriptive concentration: mean resultant length -> kappa_hat
# (Fisher), full trajectory, grouped by treatment.
kappa_df = posthoc.kappa_by_tag_full_trajectory(track_w_treat)
kappa_df.head()


$\hat\kappa$ (concentration) by treatment, median with percentile bands.


In [ ]:
# Estimated population concentration kappa_hat by treatment (median with
# percentile bands over the per-tag estimates).
plots.plot_kappa_hat(kappa_df,
                     save_path="figures/bats/kappa_hat_by_treatment_median_percentiles.pdf")
plt.show()


Mean resultant length $\vec{R}$ by treatment, median with percentile bands.


In [ ]:
# Mean resultant length R by treatment (median with percentile bands
# over the per-tag estimates).
plots.plot_rbar(kappa_df,
                save_path="figures/bats/rbar_by_treatment_median_percentiles.pdf")
plt.show()


In [ ]:
# Per-tag alignment of the animal's mean bearing to the goal direction
# (0 deg), full trajectory, grouped by treatment.
align_df = posthoc.alignment_by_tag_full_trajectory(track_w_treat)
align_df.head()


Alignment of individual mean bearings to $0^\circ$ by treatment, median with percentile bands.


In [ ]:
# Alignment of each animal's mean bearing to the goal direction by
# treatment (median with percentile bands).
plots.plot_alignment(align_df,
                     save_path="figures/bats/alignment0_by_treatment_median_percentiles.pdf")
plt.show()


## 6. Single von Mises treatment model (baseline, Bambi)

Build the modeling table for a **single** von Mises likelihood: response `HeadingRadians` (the
goal-relative bearing wrapped to $(-\pi, \pi]$), a **treatment** fixed effect, and a **per-tag random
intercept** (repeated measures). Predictors (`z_log_dist`) are standardized. This is the
non-mixed baseline; the mixture below will be judged against it.


In [ ]:
# Modeling table for the vM fit: the usable rows, the wrapped
# goal-relative heading (HeadingRadians in (-pi, pi]), categorical
# tag/treatment and the standardized log distance (prints the design
# summary).
df_track = posthoc.build_modeling_table(track_w_treat)


Specified and fit in Bambi: `HeadingRadians ~ 1 + treatment + (1|tag)` for the mean direction, and the
matching `kappa ~ 1 + treatment + (1|tag)` submodel for concentration, using the unwrap
`circ_mod` (modulo) and `softplus` links. Fit with 2000+2000 draws, `target_accept=0.9`.


In [ ]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
import matplotlib.pyplot as plt

from unwrap.link.numpy import inverse_softplus
from unwrap.prior import CircularUniformPrior
from unwrap.link.bambi import LINKS

# ============================================================
# AFTER localization only + Bambi VonMises GLMM
# Fixed effects: treatment (categorical)
# Random effects: random intercepts by tag ONLY (no random slopes)
# Uses YOUR custom links: LINKS["circ_mod"] and LINKS["softplus"]
# Priors: same idea as your other model for slopes (fixed effects),
#         but NO random effects for slopes.
# ============================================================

# -------------------------
# 2) Model formula
#   - treatment as fixed effect in BOTH mu and kappa
#   - random intercept by tag ONLY
# -------------------------
formula = bmb.Formula(
    "HeadingRadians ~ 1 + treatment + (1 | tag)",  # mu submodel
    "kappa ~ 1 + treatment + (1 | tag)",           # kappa submodel
)

# -------------------------
# 3) Priors
# -------------------------
priors = {
    # ---- mu (mean direction) ----
    "Intercept": bmb.Prior("VonMises", mu=0, kappa=1.0),

    # treatment fixed effects on mu (all dummies share this prior)
    "treatment": bmb.Prior("VonMises", mu=0, kappa=2.0),

    # random intercept on mu.
    # Original intent: tight circular (VonMises, kappa≈30 ⇒ angular sd ≈ 1/sqrt(30) ≈ 0.18) RE.
    # bambi's non-centered group-specific terms only support Normal priors, so we
    # use the small-angle Normal analogue on the pre-link scale, with a HalfNormal
    # hyperprior centred at the same angular sd (~0.18). The circular_modulo link
    # wraps the linear predictor back to the circle, so the RE still enters mu as
    # a per-animal angular shift (matching the kappa RE below).
    "1|tag": bmb.Prior(
        "Normal", mu=0.0,
        sigma=bmb.Prior("HalfNormal", sigma=1.0 / np.sqrt(30.0)),
    ),

    # ---- kappa (concentration) submodel ----
    "kappa": {
        "Intercept": bmb.Prior("Normal", mu=inverse_softplus(3), sigma=1.0),

        # treatment fixed effects on kappa (all dummies share this prior)
        "treatment": bmb.Prior("Normal", mu=0, sigma=1.0),

        # random intercept on kappa
        "1|tag": bmb.Prior("Normal", mu=0, sigma=bmb.Prior("HalfNormal", sigma=1.0)),
    },
}

# -------------------------
# 4) Build + fit (with YOUR links)
# -------------------------
model = bmb.Model(
    formula,
    df_track,
    family="vonmises",
    priors=priors,
    link={"mu": LINKS["circ_mod"], "kappa": LINKS["softplus"]},
    center_predictors=False,
    categorical=["tag", "treatment"],
    dropna=True,
)

print(model)

idata_treatment = model.fit(tune=2000, draws=2000, target_accept=0.9, random_seed=SEED, progressbar=False)

# The installed bambi takes no idata_kwargs; attach the per-observation
# log-likelihood in place (needed for the PSIS-LOO comparison below).
model.compute_log_likelihood(idata_treatment)


### Diagnostics and treatment coefficients

ArviZ summary of the vM treatment fit; the following cells flag any variables with poor ESS or
$\hat{R}$.


In [ ]:
summary_df = az.summary(idata_treatment)


In [ ]:
summary_df[(summary_df["ess_bulk"]<400) | (summary_df["ess_tail"]<400)]


In [ ]:
summary_df[(summary_df["r_hat"]>1.01)]

In [ ]:
summary_df

Treatment coefficients for the mu and kappa submodels (vM baseline).


In [ ]:

# Treatment coefficients of the vM fit: the mu (mean direction) and
# kappa (concentration) submodels with 95% HDI, control intercept on
# top.
plots.plot_treatment_coefficients_mu_kappa(idata_treatment, df_track,
                                           save_path="figures/bats/treatment_coefficients_mu_kappa.pdf")
plt.show()


Posterior predictive density at the goal direction ($0^\circ$, i.e. straight toward the goal) by
treatment for the vM baseline.


In [ ]:

# Posterior predictive density at the goal direction (0 deg) by
# treatment for the vM baseline (mean + 95% HDI over posterior draws).
plots.plot_treatment_pred_density_at0_mu_kappa(idata_treatment, df_track,
                                               save_path="figures/bats/treatment_pred_density_at0_mu_kappa.pdf")
plt.show()


### PSIS-LOO comparison (vMU vs vM)

Helper for turning a Normal on the logit scale into sigmoid-space percentiles (used to express the
mixture weight $p$ in interpretable $[0,1]$ terms), and to compute the out-of-sample comparison.


In [ ]:
# Sigmoid-space percentiles of a Normal on the logit scale:
# P = sigmoid(Normal(logit(0.9), 1)) — how a posterior mixture weight p
# with that prior/uncertainty reads on the [0, 1] scale.
from scipy.special import logit

qs, pqs = posthoc.sigmoid_normal_percentiles(logit(0.9), 1.0)
for q, p in zip(qs, pqs):
    print(f"{q:>5.2%}: {p:.6f}")


In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt

from unwrap.link.pymc import circular_modulo  # wraps angles to (-pi, pi] or similar
from scipy.special import logit

LOG_UNIF = -np.log(2*np.pi)


# -------------------------
# Data prep (shared)
# -------------------------
y = df_track["HeadingRadians"].to_numpy()

# tag indices
tag_codes, tag_idx = np.unique(df_track["tag"].to_numpy(), return_inverse=True)
n_tag = len(tag_codes)

# treatment indices (first category is reference)
treat_cat = pd.Categorical(df_track["treatment"])
treat_idx = treat_cat.codes
treat_levels = list(treat_cat.categories)
n_treat = len(treat_levels)

def inverse_softplus(x):
    return np.log(np.exp(x) - 1.0)

LOG_UNIF = -np.log(2*np.pi)


def fit_vmum_model_full(
    *,
    include_treatment_in_p: bool = True,
    include_treatment_in_mu: bool = True,
    include_treatment_in_kappa: bool = True,
    seed: int = 123,
):
    """
    vMU mixture:
      Heading ~ p * VonMises(mu, kappa) + (1-p) * VonMises(0, kappa_uniform)

    Adds optional treatment effects on:
      - p (mixture weight) via logit(p)
      - mu (goal-directed mean direction)
      - kappa (goal-directed concentration, via softplus link)

    Tag random intercepts are included for mu, kappa, and p (same as your prior versions).
    """

    with pm.Model() as vmum_full:

        # Data containers
        y_obs = pm.Data("y_obs", y)
        tag = pm.Data("tag", tag_idx)
        treat = pm.Data("treat", treat_idx)

        # ============================================================
        # 1) mu submodel (optionally treatment-dependent)
        #    mu = wrap(mu0 + mu_treat[treat] + mu_tag[tag])
        # ============================================================
        mu0 = pm.VonMises("mu0", mu=0.0, kappa=1.0)

        # tag random effect on mu (angular)
        mu_tag_kappa = pm.LogNormal("mu_tag_kappa", mu=np.log(30.0), sigma=0.1)
        mu_tag = pm.VonMises("mu_tag", mu=0.0, kappa=mu_tag_kappa, shape=n_tag)

        if include_treatment_in_mu and (n_treat > 1):
            # treatment contrasts on the circle: (n_treat-1) angular offsets vs ref
            mu_treat = pm.VonMises("mu_treat", mu=0.0, kappa=2.0, shape=n_treat - 1)
            mu_treat_full = pt.concatenate([pt.zeros((1,)), mu_treat])  # ref = 0
            mu = pm.Deterministic("mu", circular_modulo(mu0 + mu_treat_full[treat] + mu_tag[tag]))
        else:
            mu = pm.Deterministic("mu", circular_modulo(mu0 + mu_tag[tag]))

        # ============================================================
        # 2) kappa submodel (optionally treatment-dependent)
        #    eta_kappa = eta_kappa0 + kappa_treat[treat] + eta_kappa_tag[tag]*sigma
        #    kappa = softplus(eta_kappa)
        # ============================================================
        eta_kappa0 = pm.Normal("eta_kappa0", mu=inverse_softplus(5.0), sigma=1.0)

        sigma_kappa_tag = pm.HalfNormal("sigma_kappa_tag", sigma=1.0)
        eta_kappa_tag = pm.Normal("eta_kappa_tag", mu=0.0, sigma=1.0, shape=n_tag)

        eta_kappa_base = eta_kappa0 + eta_kappa_tag[tag] * sigma_kappa_tag

        if include_treatment_in_kappa and (n_treat > 1):
            # treatment contrasts on eta_kappa (real-valued)
            kappa_treat = pm.Normal("kappa_treat", mu=0.0, sigma=1.0, shape=n_treat - 1)
            kappa_treat_full = pt.concatenate([pt.zeros((1,)), kappa_treat])
            eta_kappa = eta_kappa_base + kappa_treat_full[treat]
        else:
            eta_kappa = eta_kappa_base

        kappa = pm.Deterministic("kappa", pt.softplus(eta_kappa))

        # ============================================================
        # 3) Mixture weight p (optionally treatment-dependent)
        #    logit(p) = logit_p0 + beta_treat[treat] + logit_p_tag[tag]*sigma
        #    w = sigmoid([logit_p, -logit_p])  (2-component simplex)
        # ============================================================
        logit_p0 = pm.Normal("logit_p0", mu=logit(0.8), sigma=1.0)

        sigma_p_tag = pm.HalfNormal("sigma_p_tag", sigma=1.0)
        logit_p_tag = pm.Normal("logit_p_tag", mu=0.0, sigma=1.0, shape=n_tag)

        logit_p_base = logit_p0 + logit_p_tag[tag] * sigma_p_tag

        if include_treatment_in_p and (n_treat > 1):
            beta_treat = pm.Normal("beta_treat", mu=0.0, sigma=1.0, shape=n_treat - 1)
            beta_treat_full = pt.concatenate([pt.zeros((1,)), beta_treat])
            logit_p = logit_p_base + beta_treat_full[treat]
        else:
            logit_p = logit_p_base

        w = pm.Deterministic("p_goal", pm.math.sigmoid(pt.stack([logit_p, -logit_p], axis=1)))

        # ============================================================
        # 4) Likelihood: mixture of goal-directed VM + near-uniform VM
        # ============================================================
        # kappa_uniform = 1e-6

        comp_goal = pm.VonMises.dist(mu=mu, kappa=kappa)
        # comp_unif = pm.VonMises.dist(mu=0.0, kappa=kappa_uniform)
        comp_unif = pm.DensityDist.dist(
            logp=lambda v: pt.full_like(v, LOG_UNIF),
        )

        pm.Mixture(
            "Heading_obs",
            w=w,
            comp_dists=[comp_goal, comp_unif],
            observed=y_obs,
        )

        idata = pm.sample(
            tune=2000,
            draws=2000,
            target_accept=0.9,
            random_seed=seed,
            return_inferencedata=True,
            idata_kwargs={"log_likelihood": True},
        )

    return idata, vmum_full


# ======================
# EXAMPLES
# ======================

# 1) Full: treatment effects on p, mu, and kappa
idata_mix_full, model_full = fit_vmum_model_full(
    include_treatment_in_p=True,
    include_treatment_in_mu=True,
    include_treatment_in_kappa=True,
    seed=123,
)

print("Treatment levels (reference first):", treat_levels)
print("Full model vars:", list(idata_mix_full.posterior.data_vars))


### Fit the full von Mises--Uniform mixture (PyMC)

The centerpiece. `fit_vmum_model_full` specifies **each step** as
$\text{Heading} \sim p\,\mathrm{VM}(\mu, \kappa) + (1-p)\,\mathrm{Uniform}$, with

- $\mu$ = goal-directed mean direction (population `mu0` + per-tag von Mises offsets + treatment
  offsets),
- $\kappa$ = goal-directed concentration (softplus-linked, per-tag offsets + treatment offsets),
- $p$ = probability a step is goal-directed (logit-linked, per-tag offsets + treatment offset).

The undirected (uniform) component explicitly represents exploratory movement, so a single
concentration no longer has to absorb both dispersion during homing and wandering — and no
post-treatment "homing" filter is needed.


In [ ]:
import arviz as az
import numpy as np

# -------------------------
# Convergence checks
# -------------------------
# az.summary returns r_hat, ess_bulk, ess_tail for each parameter by default.
# Use var_names to focus on the key parameters (and avoid huge outputs from deterministics).
vars_to_check = [
    "mu0",
    "mu_tag_kappa",
    "mu_tag",
    "mu_treat",
    "eta_kappa0",
    "sigma_kappa_tag",
    "eta_kappa_tag",
    "kappa_treat",
    "logit_p0",
    "sigma_p_tag",
    "logit_p_tag",
    "beta_treat",
]

# Keep only those that exist in this model run (depends on include_treatment_* flags)
present = [v for v in vars_to_check if v in idata_mix_full.posterior.data_vars]

summ = az.summary(
    idata_mix_full,
    var_names=present,
    round_to=3,
    kind="all",
)

print(summ)

# Flag problematic parameters
bad_rhat = summ.loc[summ["r_hat"] > 1.01].sort_values("r_hat", ascending=False)
bad_ess  = summ.loc[(summ["ess_bulk"] < 400) | (summ["ess_tail"] < 400)].sort_values(
    ["ess_bulk", "ess_tail"], ascending=True
)

print("\n==============================")
print("Convergence flags")
print("==============================")
print(f"Total params in summary: {len(summ)}")

if len(bad_rhat):
    print("\nRHAT > 1.01:")
    print(bad_rhat[["r_hat", "ess_bulk", "ess_tail"]])
else:
    print("\nRHAT > 1.01: none ✅")

if len(bad_ess):
    print("\nESS bulk or tail < 400:")
    print(bad_ess[["r_hat", "ess_bulk", "ess_tail"]])
else:
    print("\nESS bulk or tail < 400: none ✅")


Sampling summary for the vMU fit (check ESS / $\hat{R}$ before interpreting).


In [ ]:
# PSIS-LOO ELPD of both fitted models, best first (per-observation
# leave-one-out log-likelihoods from each model's idata).
models = {
    "von Mises-Uniform": idata_mix_full,
    "von Mises": idata_treatment,
}

res_total, best_model, loo_cache = posthoc.compute_psisloo_elpd_summary(
    models=models,
    df_fit=df_track,
    var_name=None,
    pareto_thresh=0.7,
    order_best_first=True,
)

print("Best model:", best_model)
print(res_total)


In [ ]:
# Total PSIS-LOO ELPD by model (vMU vs vM), best first.
plots.plot_total_elpd(
    res_total,
    title=None,
    xlabel="ELPD",
    vline_at_best=False,
    save_path="figures/bats/psisloo_total_elpd_all_models.pdf",
)
plt.show()


Compute **$\Delta$ELPD = vMU $-$ vM** with PSIS-LOO, total and stratified by treatment, with the SE of
the total. A positive value means the mixture predicts goal-relative headings better out of sample;
it is the evidence for a non-trivial undirected component.


In [ ]:
# Delta ELPD (vMU - vM), total and stratified by treatment, with the SE
# of the total.
delta_df = posthoc.elpd_diff_total_and_by_treatment(
    idata_vmu=idata_mix_full,      # von Mises-Uniform
    idata_vm=idata_treatment,      # von Mises
    df_fit=df_track,
    treat_col="treatment",
    treat_name_col="treatmentName",
    var_name=None,
)

treatment_order = (
    df_track["treatment"].cat.categories.tolist()
    if hasattr(df_track["treatment"], "cat")
    else sorted(delta_df.loc[delta_df["row_type"] == "treatment", "treatment"].dropna().unique())
)
delta_df


In [ ]:
# Delta ELPD (vMU - vM) total row on top of the within-treatment rows.
plots.plot_delta_total_and_by_treatment(
    delta_df,
    treatment_order=treatment_order,
    title=None,
    xlabel="ELPD difference",
    vline_at_zero=True,
    save_path="figures/bats/psisloo_delta_total_and_by_treatment_vMU_minus_vM.pdf",
)
plt.show()


Total ELPD by model (vMU vs vM), best first.


In [ ]:
# Combined two-panel figure: total ELPD by model (top) next to the
# within-treatment delta ELPD bracket (bottom).
plots.plot_psisloo_combined(res_total, delta_df, treatment_order,
                            save_path="figures/bats/psisloo_combined.pdf")
plt.show()


Pointwise $\Delta$ELPD (vMU $-$ vM) by animal and by treatment — the vMU advantage can be concentrated
in a subset of animals.


In [ ]:
# Pointwise delta ELPD (vMU - vM) per animal, grouped by treatment —
# the vMU advantage can be concentrated in a subset of animals.
res_ind = posthoc.delta_by_individual_treatment(
    idata_vmu=idata_mix_full,
    idata_vm=idata_treatment,
    df_fit=df_track,
    individual_col="tag",
    treat_col="treatment",
    treat_name_col="treatmentName",
    var_name=None,
)

treatment_order = (
    df_track["treatment"].cat.categories.tolist()
    if hasattr(df_track["treatment"], "cat")
    else sorted(res_ind["treatment"].unique())
)


In [ ]:
# Pointwise delta ELPD (vMU - vM) by animal and by treatment (animals
# sorted by their own delta ELPD within each panel).
plots.plot_delta_subplots_by_treatment(
    res_ind,
    treatment_order=treatment_order,
    save_path="figures/bats/psisloo_delta_by_individual_by_treatment_vMU_minus_vM.pdf",
)
plt.show()


## 7. vMU population-level summaries

Population-level vMU summaries: per-treatment polar densities of the goal-relative bearings, showing
the directed + undirected mixture.


In [ ]:
# Per-treatment population polar densities of the goal-relative
# bearings for the vMU fit: the directed + undirected mixture.
plots.plot_mixture_polar_pdfs(idata_mix_full, df_track,
                              save_path="figures/bats/vmum_mixture_full_polar_pdfs_population_only.pdf")
plt.show()


Per-treatment population **mixture weight** $p$ (probability a step is goal-directed), goal-directed
alignment to $0^\circ$, and goal-directed $\kappa$, each with posterior means and HDIs.


In [ ]:
# Population-level vMU summaries by treatment: mixture weight p,
# goal-directed mean direction and concentration, and the predictive
# density / mass in the goal window (posterior mean + 95% HDI).
res_full = posthoc.summarize_pop_p_kappa_mu_density0_mass45_alignment0_by_treatment(
    idata_mix_full=idata_mix_full,
    df_fit=df_track,
    treat_col="treatment",
    treat_name_col="treatmentName",
    hdi_prob=0.95,
    max_draws=2000,
    kappa_uniform=posthoc.KAPPA_UNIFORM,
    mass_half_width_deg=45.0,
)

# Shared x-limit so the mixture and directed-density panels are
# directly comparable.
dens_xmax = float(max(res_full["dens0_mix_hi"].max(), res_full["dens0_dir_hi"].max()) * 1.05)
dens_xlim = (0.0, dens_xmax)


In [ ]:
# Mixture weight p (probability a step is goal-directed) by treatment;
# dashed line at the uniform 0.5.
plots.plot_mean_hdi_horizontal(
    res_full, "p_mean", "p_lo", "p_hi", "Mixture weight",
    xlim=(0, 1), vline_x=0.5,
    show_treatment_labels=False, show_treatment_ticks=False,
    save_path="figures/bats/fullmodel_mixture_weight_by_treatment_mean_hdi.pdf",
)
plt.show()


In [ ]:

# Goal-directed concentration kappa by treatment.
plots.plot_mean_hdi_horizontal(
    res_full, "kappa_mean", "kappa_lo", "kappa_hi", "Concentration " + rf"($\kappa$)",
    show_treatment_labels=False, show_treatment_ticks=False,
    save_path="figures/bats/fullmodel_kappa_by_treatment_mean_hdi.pdf",
)
plt.show()


In [ ]:
# Predictive density at the goal direction (full mixture) by treatment;
# dashed line at the exact uniform density 1/(2 pi).
plots.plot_mean_hdi_horizontal(
    res_full, "dens0_mix_mean", "dens0_mix_lo", "dens0_mix_hi",
    "Predictive density (mixture)",
    xlim=dens_xlim, vline_x=1.0 / (2.0 * np.pi),
    show_treatment_labels=False, show_treatment_ticks=False,
    save_path="figures/bats/fullmodel_predictive_density_at_0_mixture_by_treatment_mean_hdi.pdf",
)
plt.show()


In [ ]:
# Predictive density at the goal direction (directed component only) by
# treatment; the uniform line stays as the no-preference reference.
plots.plot_mean_hdi_horizontal(
    res_full, "dens0_dir_mean", "dens0_dir_lo", "dens0_dir_hi",
    "Predictive density (directed component)",
    xlim=dens_xlim, vline_x=1.0 / (2.0 * np.pi),
    show_treatment_labels=False, show_treatment_ticks=False,
    save_path="figures/bats/fullmodel_predictive_density_at_0_directed_component_by_treatment_mean_hdi.pdf",
)
plt.show()


In [ ]:
# Predictive mass within +/-45 deg of the goal direction by treatment;
# dashed line at the uniform 0.25.
plots.plot_mean_hdi_horizontal(
    res_full, "mass45_mean", "mass45_lo", "mass45_hi", "Predictive mass (±45°)",
    xlim=(0, 1), vline_x=0.25,
    show_treatment_labels=False, show_treatment_ticks=False,
    save_path="figures/bats/fullmodel_predictive_mass_pm45deg_by_treatment_mean_hdi.pdf",
)
plt.show()


In [ ]:
# Alignment to the goal direction E[cos(theta)] under the mixture by
# treatment.
plots.plot_mean_hdi_horizontal(
    res_full, "align0_mean", "align0_lo", "align0_hi", "Alignment",
    xlim=(0, 1),
    show_treatment_labels=False, show_treatment_ticks=False,
    save_path="figures/bats/fullmodel_alignment_to_zero_by_treatment_mean_hdi.pdf",
)
plt.show()


Combined 2x2 summary plot of the out-of-sample comparison against the population-level vMU
parameters.


In [ ]:
# Combined summary figure: delta ELPD (top) next to the population vMU
# parameters in a 2x2 block (mean angle, mixture weight, MRL,
# predictive density at the goal).
plots.plot_psisloo_plus_params_2x2(
    delta_df, treatment_order, idata_mix_full, df_track,
    save_path="figures/bats/psisloo_plus_params_2x2.pdf",
)
plt.show()


Per-treatment predicted posterior density at the goal direction for the vMU (analog of the vM
baseline figure above).


In [ ]:
# Matched two-panel comparison of the predictive density at the goal
# direction: vMU directed component (top) vs vM (bottom); the uniform
# dashed line is the no-preference reference.
plots.plot_predictive_density_vmu_directed_vs_vm(
    res_full, df_track, idata_treatment,
    save_path="figures/bats/predictive_density_two_model_comparison_matched_vmu_directed_vs_vm.pdf",
)
plt.show()


In [ ]:
# Three-panel summary: total ELPD by model, delta ELPD by treatment,
# and the vMU mixture predictive density at the goal direction.
plots.plot_psisloo_plus_predictive_density_3panel(
    res_total, delta_df, treatment_order, res_full,
    save_path="figures/bats/psisloo_plus_predictive_density_3panel.pdf",
)
plt.show()


## 8. Per-animal level

Helpers plus **per-animal (tag-level)** vMU summaries: each bat's mixture weight, goal-directed
alignment, and $\kappa$. This is where pronounced between-animal heterogeneity becomes visible.


In [ ]:
# Per-animal (tag-level) vMU summaries from the model terms: mixture
# weight p, goal-directed kappa and density at the goal, alignment to
# 0 deg — plus the population-only table that draws the reference band.
res_ind, treatment_order = posthoc.summarize_taglevel_from_model_terms(
    idata_mix_full=idata_mix_full,
    df_fit=df_track,
    treat_col="treatment",
    treat_name_col="treatmentName",
    individual_col="tag",
    hdi_prob=0.95,
    max_draws=2000,
    kappa_uniform=posthoc.KAPPA_UNIFORM,
)

pop_df = posthoc.pop_summary_by_treatment_from_model_terms(
    idata_mix_full=idata_mix_full,
    df_fit=df_track,
    treat_col="treatment",
    treat_name_col="treatmentName",
    hdi_prob=0.95,
    max_draws=2000,
    kappa_uniform=posthoc.KAPPA_UNIFORM,
)

# reference values for the dashed lines
uniform_density_at_0 = 1.0 / (2.0 * np.pi)
mixture_weight_ref = 0.5


In [ ]:
# Per-animal mixture weight by treatment (gray population band + solid
# population mean, dashed line at 0.5; animals sorted within each
# panel).
plots.plot_individual_subplots_by_treatment(
    res_ind, treatment_order,
    value_col="p_mean", lo_col="p_lo", hi_col="p_hi",
    xlabel="Mixture weight",
    pop_df=pop_df, pop_mean_col="p_mean", pop_lo_col="p_lo", pop_hi_col="p_hi",
    xlim=(0, 1),
    ref_vline_x=mixture_weight_ref,
    save_path="figures/bats/fullmodel_taglevel_mixture_weight_by_treatment_mean_hdi.pdf",
)
plt.show()


In [ ]:

# Per-animal goal-directed concentration kappa by treatment.
plots.plot_individual_subplots_by_treatment(
    res_ind, treatment_order,
    value_col="kappa_mean", lo_col="kappa_lo", hi_col="kappa_hi",
    xlabel="Concentration " + rf"($\kappa$)",
    pop_df=pop_df, pop_mean_col="kappa_mean", pop_lo_col="kappa_lo", pop_hi_col="kappa_hi",
    save_path="figures/bats/fullmodel_taglevel_kappa_by_treatment_mean_hdi.pdf",
)
plt.show()


In [ ]:
# Per-animal predictive density at the goal direction by treatment
# (dashed line at the uniform density).
plots.plot_individual_subplots_by_treatment(
    res_ind, treatment_order,
    value_col="dens0_mean", lo_col="dens0_lo", hi_col="dens0_hi",
    xlabel="Predictive density",
    pop_df=pop_df, pop_mean_col="dens0_mean", pop_lo_col="dens0_lo", pop_hi_col="dens0_hi",
    ref_vline_x=uniform_density_at_0,
    save_path="figures/bats/fullmodel_taglevel_predictive_density_at_0_by_treatment_mean_hdi.pdf",
)
plt.show()


In [ ]:
# Per-animal alignment to the goal direction by treatment.
plots.plot_individual_subplots_by_treatment(
    res_ind, treatment_order,
    value_col="align0_mean", lo_col="align0_lo", hi_col="align0_hi",
    xlabel="Alignment",
    pop_df=pop_df, pop_mean_col="align0_mean", pop_lo_col="align0_lo", pop_hi_col="align0_hi",
    xlim=(0, 1),
    save_path="figures/bats/fullmodel_taglevel_alignment_to_zero_by_treatment_mean_hdi.pdf",
)
plt.show()


In [ ]:
# Population-level PPC by treatment: hierarchical bootstrap of the
# observed goal-relative headings vs the same summaries simulated from
# the vMU posterior (tag random effects included); the heavy machinery
# runs in goldshtein_bats_posthoc.
N_KDE_POINTS = 360       # evaluation grid for the KDE
N_HIST_BINS = 16
N_BOOT_OBS = 1000        # hierarchical bootstrap replicates (observed)
N_PPC_DRAWS = 4000       # posterior draws for the PPC
PPC_BOOT_PER_DRAW = 1
KDE_KAPPA = 30.0
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
RNG_SEED = 123

df_ppc = df_track.reset_index(drop=True).copy()  # index must be 0..N-1 for PPC slicing
treat_levels, groups = posthoc.build_groups(
    df_ppc, y_col="HeadingRadians", tag_col="tag", treat_col="treatment"
)
treatment_label_map = (
    df_ppc.groupby("treatment", observed=False)["treatmentName"].first().to_dict()
)

angles = np.linspace(-np.pi, np.pi, N_KDE_POINTS, endpoint=False)
bin_edges = np.linspace(-np.pi, np.pi, N_HIST_BINS + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

obs_boot = posthoc.bootstrap_observed(
    groups=groups, angles=angles, bin_edges=bin_edges,
    n_boot=N_BOOT_OBS, kde_kappa=KDE_KAPPA, n_jobs=N_JOBS, seed=RNG_SEED,
)

ppc = posthoc.posterior_predictive_summaries(
    idata=idata_mix_full, df=df_ppc, treat_levels=treat_levels,
    angles=angles, bin_edges=bin_edges, kde_kappa=KDE_KAPPA,
    n_draws=N_PPC_DRAWS, n_boot_within_draw=PPC_BOOT_PER_DRAW,
    n_jobs=N_JOBS, seed=RNG_SEED,
)


In [ ]:
# Population PPC by treatment: observed hierarchical-bootstrap KDE vs
# posterior predictive KDE of the goal-relative bearings.
plots.plot_ppc_by_treatment(
    treat_levels, angles, bin_centers, obs_boot, ppc,
    treatment_label_map=treatment_label_map,
    save_path="figures/bats/bat_population_ppc_by_treatment.pdf",
)
plt.show()


### Posterior predictive checks

Build the PPC machinery (population and individual): resample / bootstrap the goal-relative headings
under the vMU, compare to the observed, per treatment and per animal.


In [ ]:
# Individual PPC: per-animal bootstrap (observed) vs posterior KDE of
# the goal-directed component; the grid is built in posthoc and drawn
# in the plotting script.
ppc_grid = posthoc.individual_ppc(idata_mix_full, df_track, n_cols=8)


Individual PPC: bootstrap vs posterior KDE of the goal-directed component per animal.


In [ ]:
# Individual PPC grid: per-animal data bootstrap vs posterior
# predictive KDE of the goal-directed component.
plots.plot_individual_ppc_grid(ppc_grid,
                               save_path="figures/bats/bat_individual_ppc_bootstrap_vs_ppc_kde.pdf")
plt.show()


In [ ]:
# Posterior probability that each step is goal-directed (p_goal):
# responsibilities of the two mixture components per step, summarized
# over 2000 draws with the exact uniform-circle likelihood; the p
# columns are appended to df_track and joined onto the tracks in the
# figures below.
df_track["_step"] = df_track.groupby("tag", observed=False).cumcount()
df_track, _ = posthoc.append_pgoal_post_to_df(
    df_track, idata_mix_full, y_col="HeadingRadians", max_draws=2000, hdi_prob=0.95
)

print(df_track[["tag", "treatment", "_step", "p_goal_post_mean", "p_goal_post_lo", "p_goal_post_hi"]].head())


## 9. Goal-directedness along the trajectory

Posterior probability that each step belongs to the **directed (goal-seeking) component**
($p_{goal}$), plotted along each trajectory (by distance / progress toward the goal). Directedness is
expected to be lower early in a flight (localization / exploration) and higher near the endpoint
(homing).


In [ ]:
# Tracks colored by the per-step goal-directed probability (viridis),
# one panel per treatment.
plots.plot_tracks_pgoal_viridis(
    track_w_treat,
    p_col="p_goal_post_mean",
    df_prob=df_track,
    prob_tag_col="tag",
    prob_step_col="_step",
    prob_p_col="p_goal_post_mean",
    gap_col="gap_before",
    gap_linestyle="--",
    gap_alpha=0.80,
    seg_lw=1.0,
    wspace=0.10,
    hspace=0.10,
    fig_w=8.0,
    fig_h=6,
    save_path="figures/bats/treatment_panels_tracks_pgoal.pdf",
)
plt.show()


In [ ]:
# Compact version of the p_goal-colored tracks (smaller figure, bigger
# colorbar text, track layers rasterized inside the saved PDF).
plots.plot_tracks_pgoal_small_bigfont(
    track_w_treat,
    p_col="p_goal_post_mean",
    df_prob=df_track,
    prob_tag_col="tag",
    prob_step_col="_step",
    prob_p_col="p_goal_post_mean",
    gap_col="gap_before",
    gap_linestyle="--",
    gap_alpha=0.75,
    seg_lw=0.8,
    wspace=0.03,
    hspace=0.02,
    save_path="figures/bats/treatment_panels_tracks_pgoal_small_bigfont.pdf",
)
plt.show()


Population-level $p_{goal}$ as a function of distance from the flight's endpoint — lower early, higher
as the animal approaches its goal.


In [ ]:
# Population-level p_goal along the trajectory, kernel-smoothed per tag
# then averaged per draw (per-tag truncation to the observed range,
# post-smoothing across x, masking where few tags contribute).
# df_track_with_p, resp and the curves below are also reused by the
# compact and per-animal cells further down.
STEP_LENGTH_M = 60.0
DIST_COL = "dist_to_dest_grad_m"
LAST_N_STEPS = 110

df_track_with_p, resp = posthoc.append_pgoal_post_to_df(
    df_track, idata_mix_full, y_col="HeadingRadians", max_draws=2000, hdi_prob=0.95
)
print(df_track_with_p[["p_goal_post_mean", "p_goal_post_lo", "p_goal_post_hi"]].head())

treat_levels = (
    list(df_track_with_p["treatment"].cat.categories)
    if isinstance(df_track_with_p["treatment"].dtype, pd.CategoricalDtype)
    else sorted(df_track_with_p["treatment"].dropna().unique().tolist())
)

# A) Distance-to-destination
_, out_dist_k = posthoc.population_curve_kernel_per_tag(
    df_track_with_p, resp,
    x_kind="distance",
    dist_col=DIST_COL,
    n_grid=2000,
    bandwidth=120.0,
    chunk_size=4000,
    hdi_prob=0.95,
    truncate_to_data_range=True,
    truncate_by_weight=True,
    den_frac=0.01,
    pop_post_smooth=True,
    pop_smooth_bandwidth=240.0,
    min_tags_at_x=2,
)

# B) Steps-from-end (x in meters)
_, out_steps_k = posthoc.population_curve_kernel_per_tag(
    df_track_with_p, resp,
    x_kind="steps_from_end",
    last_n_steps=LAST_N_STEPS,
    steps_x_units="m",
    step_length_m=STEP_LENGTH_M,
    n_grid=2000,
    bandwidth=120.0,
    chunk_size=4000,
    hdi_prob=0.95,
    truncate_to_data_range=True,
    truncate_by_weight=True,
    den_frac=0.01,
    pop_post_smooth=True,
    pop_smooth_bandwidth=240.0,
    min_tags_at_x=2,
)


In [ ]:
# Population mean p_goal vs distance to the destination (x-axis
# inverted: approaching the goal moves right).
plots.plot_population_curve_smooth(
    out_dist_k, treat_levels,
    xlabel="Distance to destination (m)",
    ylabel="Mean directed-component probability",
    invert_x=True,
    save_path="figures/bats/p_goal_pop_kernel_distance_TRUNC_SMOOTH.pdf",
)
plt.show()


In [ ]:
# Population mean p_goal vs cumulative path distance from the end of
# the flight (negative = steps before the end).
plots.plot_population_curve_smooth(
    out_steps_k, treat_levels,
    xlabel="Cumulative path distance from end (m)",
    ylabel="Mean directed-component probability",
    invert_x=False,
    xlim=(-LAST_N_STEPS * STEP_LENGTH_M, 0),
    x_tick_abs=True,
    save_path="figures/bats/p_goal_pop_kernel_distFromEndM_TRUNC_SMOOTH.pdf",
)
plt.show()


In [ ]:

# Compact (paper-width) version: p_goal vs distance to the destination
# (same curves as above).
plots.plot_population_curve_smooth_compact(
    out_dist_k, treat_levels,
    xlabel="Distance to destination (m)",
    ylabel="Directed-component\nprobability",
    invert_x=True,
    save_path="figures/bats/p_goal_pop_kernel_distance_TRUNC_SMOOTH_compact.pdf",
)
plt.show()


In [ ]:

# Compact version: p_goal vs cumulative path distance from the end of
# the flight (same curves as above).
plots.plot_population_curve_smooth_compact(
    out_steps_k, treat_levels,
    xlabel="Cumulative path distance from end (m)",
    ylabel="Directed-component\nprobability",
    invert_x=False,
    xlim=(-LAST_N_STEPS * STEP_LENGTH_M, 0),
    x_tick_abs=True,
    save_path="figures/bats/p_goal_pop_kernel_distFromEndM_TRUNC_SMOOTH_compact.pdf",
)
plt.show()


Per-animal (faceted) $p_{goal}$ along the flight, showing the heterogeneity in when each bat commits
to goal-directed travel.


In [ ]:
# Per-animal p_goal curves (kernel-smoothed per tag, truncated to each
# tag's data range): the faceted panels show when each bat commits to
# goal-directed travel.
_, out_tags_dist = posthoc.tag_curves_kernel_by_treatment(
    df_track_with_p, resp,
    x_kind="distance",
    dist_col=DIST_COL,
    n_grid=2000,
    bandwidth=120.0,
    chunk_size=4000,
    hdi_prob=0.95,
    truncate_to_data_range=True,
    truncate_by_weight=True,
    den_frac=0.01,
)

_, out_tags_steps = posthoc.tag_curves_kernel_by_treatment(
    df_track_with_p, resp,
    x_kind="steps_from_end",
    last_n_steps=LAST_N_STEPS,
    steps_x_units="m",
    step_length_m=STEP_LENGTH_M,
    n_grid=2000,
    bandwidth=120.0,
    chunk_size=4000,
    hdi_prob=0.95,
    truncate_to_data_range=True,
    truncate_by_weight=True,
    den_frac=0.01,
)


In [ ]:
# Per-animal p_goal vs distance to the destination (four treatments
# per page; one saved PDF per page).
plots.plot_tag_curves_faceted_4panel(
    out_tags_dist, treat_levels,
    xlabel="Distance to destination (m)",
    ylabel="Directed-component probability",
    invert_x=True,
    fig_h=6,
    fig_w=8,
    save_path="figures/bats/p_goal_tag_kernel_distance_faceted_TRUNC.pdf",
)
plt.show()


In [ ]:
# Per-animal p_goal vs cumulative path distance from the end of the
# flight (one saved PDF per page).
plots.plot_tag_curves_faceted_4panel(
    out_tags_steps, treat_levels,
    xlabel="Cumulative path distance from end (m)",
    ylabel="Directed-component probability",
    invert_x=False,
    xlim=(-LAST_N_STEPS * STEP_LENGTH_M, 0),
    x_tick_abs=True,
    fig_h=6,
    fig_w=8,
    save_path="figures/bats/p_goal_tag_kernel_distFromEndM_faceted_TRUNC.pdf",
)
plt.show()
